# DEMANDA-IA · Fase I — Baseline de pronóstico SKU–sucursal
**Farmacias Occidente Salud** · Gestión de proyectos de inteligencia artificial · Actividad 5
Autor: José Gerardo Ortiz Leñero

Este cuaderno ejecuta de punta a punta el núcleo técnico del primer baseline:
1. Preparación del entorno (Colab o local) y versión de datos.
2. Limpieza y estandarización (`fuentes/datos_prep.py`) con reporte de calidad.
3. Análisis exploratorio para detectar patrones y anomalías.
4. Entrenamiento y ajuste de 3 modelos + referencia naive con Grid Search y validación cruzada temporal (`fuentes/train.py`), registrado en MLflow.
5. Comparación estadística de resultados y consulta de trazabilidad en MLflow.

## 1. Entorno
En Colab, la celda clona el repositorio e instala las versiones exactas de `requirements.txt` (si pip actualiza pandas, reinicia el entorno y vuelve a ejecutar). En local, basta con abrir el cuaderno desde `fuentes/` o desde la raíz.

In [ ]:
import os, sys, subprocess
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
URL_REPO = "https://github.com/<tu_usuario>/Actividad5.git"   # <-- reemplazar por la URL de tu repositorio

if EN_COLAB:
    if not Path("Actividad5").exists():
        subprocess.run(["git", "clone", URL_REPO], check=True)
    os.chdir("Actividad5")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif Path.cwd().name == "fuentes":
    os.chdir("..")

RAIZ = Path.cwd()
sys.path.insert(0, str(RAIZ / "fuentes"))
print("Raíz del repositorio:", RAIZ)

In [ ]:
import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import datos_prep as dp

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
GRAF = RAIZ / "evidencias" / "graficas"; GRAF.mkdir(parents=True, exist_ok=True)
def guardar(fig, nombre):
    fig.tight_layout(); fig.savefig(GRAF / nombre, dpi=140, bbox_inches="tight")

import sklearn, xgboost
print({m.__name__: m.__version__ for m in [pd, np, sklearn, xgboost, mlflow]})

## 2. Datos originales (`datos/datos_ini`)
Fuente pública: *Store Item Demand Forecasting Challenge* (Kaggle). Ventas diarias de 50 artículos en 10 tiendas, 2013–2017. Se usa como **proxy estructural** del problema de Farmacias Occidente Salud: misma granularidad (fecha × sucursal × SKU) y mismo objetivo (unidades diarias), mientras se obtiene el extracto del ERP real.

In [ ]:
ini = dp.cargar_datos_ini()
print(ini.shape); display(ini.head())
print("SHA-256 datos_ini:", dp.sha256_archivo(dp.RUTA_INI))
ini.info()

In [ ]:
diccionario = pd.DataFrame([
    ["date",  "fecha",       "datetime64", "Día de la venta (AAAA-MM-DD)"],
    ["store", "sucursal_id", "int16",      "Identificador de sucursal (1–10) → codigo_sucursal SUC-XX"],
    ["item",  "sku_id",      "int16",      "Identificador de artículo (1–50) → codigo_sku SKU-XXX"],
    ["sales", "unidades",    "int32",      "Unidades vendidas en el día (variable objetivo)"],
    ["—",     "es_imputado", "int8",       "1 si el valor se imputó (R7)"],
    ["—",     "es_devolucion","int8",      "1 si el valor original era negativo (R5)"],
    ["—",     "es_anomalia", "int8",       "1 si |z robusto| > 3.5 (R8); se conserva, no se elimina"],
    ["—",     "z_robusto",   "float32",    "Z-score robusto vs mediana móvil centrada de 28 días"],
], columns=["columna_origen", "columna_limpia", "tipo", "descripción"])
diccionario

## 3. Limpieza, estandarización y versión 1.1.0 (`datos/datos_limp`)
Cada regla tiene un criterio explícito y se contabiliza en `reporte_calidad.json`.

In [ ]:
for k, v in dp.REGLAS.items(): print(f"{k}: {v}")

In [ ]:
dp.main()   # genera datos_limp/ventas_sku_sucursal_limpio.csv.gz, reporte_calidad.json y MANIFIESTO_DATOS.json
rep = json.loads(dp.RUTA_REPORTE.read_text(encoding="utf-8"))
pd.Series({k: v for k, v in rep.items() if k.startswith("R") or k.startswith("filas") or k.startswith("n_")}).to_frame("valor")

El dataset público llega sin nulos, duplicados ni negativos. Para demostrar que las reglas **sí funcionan** ante una exportación real del POS, la siguiente prueba contamina deliberadamente una muestra (duplicados, conflicto de llave, nulo, devolución, días faltantes, llaves inválidas y un pico) y verifica que cada regla se active.

In [ ]:
print(subprocess.run([sys.executable, "pruebas/test_datos_prep.py"], capture_output=True, text=True).stdout)

## 4. Análisis exploratorio: patrones y anomalías

In [ ]:
df = pd.read_csv(dp.RUTA_LIMP, parse_dates=["fecha"])
total = df.groupby("fecha")["unidades"].sum()
fig, ax = plt.subplots(figsize=(11, 3.8))
ax.plot(total.index, total, lw=.6, color="#9db4c0", label="Total diario")
ax.plot(total.index, total.rolling(28, center=True).mean(), lw=2, color="#1f3b4d", label="Media móvil 28 días")
ax.set_title("Fig. 1. Unidades totales diarias (500 series SKU–sucursal)"); ax.set_ylabel("Unidades"); ax.legend()
guardar(fig, "eda_01_serie_total.png")
crec = df.groupby(df.fecha.dt.year)["unidades"].sum()
print("Crecimiento anual (%):", (crec.pct_change()*100).round(1).dropna().to_dict())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
dias = ["Lun","Mar","Mié","Jue","Vie","Sáb","Dom"]
d = df.assign(dow=df.fecha.dt.dayofweek)
sns.barplot(data=d, x="dow", y="unidades", ax=ax[0], color="#2a9d8f", errorbar=None)
ax[0].set_xticks(range(7), dias); ax[0].set_title("Fig. 2a. Unidades medias por día de la semana"); ax[0].set_xlabel("")
piv = df.assign(anio=df.fecha.dt.year, mes=df.fecha.dt.month).pivot_table(index="anio", columns="mes", values="unidades", aggfunc="mean")
sns.heatmap(piv, ax=ax[1], cmap="YlGnBu", annot=True, fmt=".0f", cbar=False, annot_kws={"size": 7})
ax[1].set_title("Fig. 2b. Unidades medias por año y mes"); ax[1].set_xlabel("Mes"); ax[1].set_ylabel("")
guardar(fig, "eda_02_estacionalidad.png")
print((d.groupby("dow").unidades.mean() / d.unidades.mean()).round(3).set_axis(dias).to_dict())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw={"width_ratios": [2.2, 1]})
m = df.pivot_table(index="sucursal_id", columns="sku_id", values="unidades", aggfunc="mean")
sns.heatmap(m, ax=ax[0], cmap="rocket_r", cbar_kws={"label": "unidades/día"})
ax[0].set_title("Fig. 3a. Demanda media por sucursal × SKU"); ax[0].set_xlabel("SKU"); ax[0].set_ylabel("Sucursal")
cv_serie = df.groupby(["sucursal_id","sku_id"]).unidades.agg(lambda s: s.std()/s.mean())
ax[1].hist(cv_serie, bins=30, color="#e76f51"); ax[1].set_title("Fig. 3b. Coef. de variación por serie"); ax[1].set_xlabel("CV")
guardar(fig, "eda_03_sucursal_sku.png")
print("Rango de demanda media por serie:", m.values.min().round(1), "a", m.values.max().round(1))
print("Correlación de perfiles SKU entre sucursales (media):", np.corrcoef(m.values)[np.triu_indices(10,1)].mean().round(4))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
an = df[df.es_anomalia == 1]
an.groupby(an.fecha.dt.month).size().reindex(range(1,13), fill_value=0).plot.bar(ax=ax[0], color="#d62828")
ax[0].set_title(f"Fig. 4a. Anomalías marcadas por mes (n={len(an):,}; {100*len(an)/len(df):.2f}%)"); ax[0].set_xlabel("Mes")
s = df[(df.sucursal_id==2) & (df.sku_id==15) & (df.fecha.dt.year==2017)]
ax[1].plot(s.fecha, s.unidades, lw=.8, color="#1f3b4d")
ax[1].scatter(s.fecha[s.es_anomalia==1], s.unidades[s.es_anomalia==1], color="#d62828", zorder=3, s=18, label="anomalía (|z|>3.5)")
ax[1].set_title("Fig. 4b. SUC-02 / SKU-015, 2017"); ax[1].legend()
guardar(fig, "eda_04_anomalias.png")
print("Anomalías por signo del z:", (np.sign(an.z_robusto)).value_counts().to_dict())

In [ ]:
def acf_simple(x, nlags):
    x = np.asarray(x, float) - np.mean(x)
    return np.array([1.0] + [np.corrcoef(x[:-k], x[k:])[0, 1] for k in range(1, nlags + 1)])
a = acf_simple(total.values, 400)
fig, ax = plt.subplots(figsize=(11, 3.2))
ax.vlines(range(len(a)), 0, a, color="#457b9d", lw=.8)
for l in (7, 14, 28, 364): ax.axvline(l, color="#e07a1f", ls="--", lw=.8)
ax.set_title("Fig. 5. Autocorrelación del total diario (líneas: rezagos 7, 14, 28 y 364)"); ax.set_xlabel("Rezago (días)")
guardar(fig, "eda_05_autocorrelacion.png")
print({l: round(a[l], 3) for l in (1, 7, 14, 28, 182, 364)})

In [ ]:
fig, ax = plt.subplots(figsize=(11, 2.2))
bloques = [("Sólo historia para lag 364", "2013-01-01", "2014-01-01", "#cccccc"),
           ("Entrenamiento (fold 1)", "2014-01-01", "2016-01-01", "#a8dadc"),
           ("Val. fold 1", "2016-01-01", "2016-07-01", "#457b9d"), ("Val. fold 2", "2016-07-01", "2017-01-01", "#1d3557"),
           ("Val. fold 3", "2017-01-01", "2017-07-01", "#264653"), ("Holdout test", "2017-07-01", "2018-01-01", "#e63946")]
for et, i, f, c in bloques:
    ax.barh(0, pd.Timestamp(f)-pd.Timestamp(i), left=pd.Timestamp(i), color=c, edgecolor="white")
    ax.text(pd.Timestamp(i) + (pd.Timestamp(f)-pd.Timestamp(i))/2, 0, et, ha="center", va="center", color="white" if c not in ("#cccccc","#a8dadc") else "black", fontsize=8)
ax.set_yticks([]); ax.set_title("Fig. 6. Esquema temporal: ventana expansiva (3 folds de 6 meses) + holdout jul–dic 2017")
guardar(fig, "eda_06_esquema_validacion.png")

## 5. Entrenamiento, Grid Search y registro en MLflow
Se ejecuta el script versionado `fuentes/train.py` (mismo código que en consola), de modo que el resultado del cuaderno y el de la línea de comandos sean idénticos. El seguimiento queda en `mlflow.db` (SQLite) y los artefactos en `mlruns/`.

In [ ]:
r = subprocess.run([sys.executable, "fuentes/train.py"], capture_output=True, text=True)
print("\n".join(l for l in r.stdout.splitlines() if "warn" not in l.lower()))
if r.returncode != 0: print(r.stderr[-3000:])

## 6. Resultados y comparación estadística

In [ ]:
met = pd.read_csv(RAIZ / "resultados" / "tabla_metricas.csv")
cols = ["modelo", "mae", "rmse", "r2", "wape", "sesgo_pct", "cv_mae_media", "cv_mae_std", "seg_entrenamiento", "mejores_hiperparametros"]
met[cols].round(4)

In [ ]:
comp = json.loads((RAIZ / "resultados" / "comparacion_estadistica.json").read_text(encoding="utf-8"))
folds = pd.DataFrame(comp["mae_cv_por_fold"], index=["fold 1 (2016-S1)", "fold 2 (2016-S2)", "fold 3 (2017-S1)"])
fig, ax = plt.subplots(1, 2, figsize=(12, 3.9))
folds.plot(marker="o", ax=ax[0]); ax[0].set_title("Fig. 7a. MAE por fold de validación (mejor combinación)"); ax[0].set_ylabel("MAE")
x = np.arange(len(met)); w = .38
ax[1].bar(x - w/2, met.mae, w, label="MAE", color="#457b9d"); ax[1].bar(x + w/2, met.rmse, w, label="RMSE", color="#e07a1f")
for i, v in enumerate(met.mae): ax[1].text(i - w/2, v + .1, f"{v:.2f}", ha="center", fontsize=8)
ax[1].set_xticks(x, met.modelo, rotation=10); ax[1].set_title("Fig. 7b. Error en holdout jul–dic 2017"); ax[1].legend()
guardar(fig, "res_07_cv_y_test.png")
print("Friedman χ² =", round(comp["friedman_chi2"], 1), " p =", comp["friedman_p"])
print("Rango promedio (1 = mejor):", {k: round(v, 2) for k, v in comp["rango_promedio_por_modelo"].items()})

In [ ]:
pred = pd.read_csv(RAIZ / "resultados" / "predicciones_test.csv.gz", parse_dates=["fecha"])
modelos = [c.replace("pred_", "") for c in pred.columns if c.startswith("pred_")]
mae_s = pd.DataFrame({m: (pred.real - pred[f"pred_{m}"]).abs().groupby([pred.codigo_sucursal, pred.codigo_sku]).mean() for m in modelos})
fig, ax = plt.subplots(1, 2, figsize=(12, 3.9))
sns.boxplot(data=mae_s, ax=ax[0], palette="Set2"); ax[0].set_title("Fig. 8a. MAE por serie SKU–sucursal (n=500)"); ax[0].tick_params(axis="x", rotation=10)
wape_suc = pd.DataFrame({m: (pred.real - pred[f"pred_{m}"]).abs().groupby(pred.codigo_sucursal).sum() / pred.real.groupby(pred.codigo_sucursal).sum() for m in modelos})
(100*wape_suc).plot.bar(ax=ax[1], width=.8); ax[1].set_title("Fig. 8b. WAPE (%) por sucursal"); ax[1].set_xlabel("")
guardar(fig, "res_08_error_por_serie.png")
mae_s.describe().round(3)

In [ ]:
pares = pd.read_csv(RAIZ / "resultados" / "comparacion_pares.csv")
pares[["comparacion", "dif_mae_media", "ic95_inf", "ic95_sup", "series_donde_A_gana", "wilcoxon_p_holm", "t_pareada_p", "efecto_d_cohen", "dm_estadistico", "dm_p"]]

In [ ]:
import matplotlib.image as mpimg
fig, ax = plt.subplots(2, 1, figsize=(11, 7))
for a, m in zip(ax, ["xgboost", "ridge"]):
    a.imshow(mpimg.imread(RAIZ / "resultados" / "graficas" / f"{m}_real_vs_pred.png")); a.axis("off")
guardar(fig, "res_09_real_vs_pred.png")

In [ ]:
import matplotlib.image as mpimg
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, m in zip(ax, ["xgboost", "random_forest"]):
    a.imshow(mpimg.imread(RAIZ / "resultados" / "graficas" / f"{m}_importancia.png")); a.axis("off")
guardar(fig, "res_10_importancia.png")

## 7. Trazabilidad en MLflow
Consulta programática de lo registrado: corridas padre por modelo, corridas hijas por combinación del grid, parámetros, métricas, etiquetas de versión (commit de Git, hash del dataset) y registro de modelos con alias `campeon`.

In [ ]:
mlflow.set_tracking_uri(f"sqlite:///{RAIZ / 'mlflow.db'}")
runs = mlflow.search_runs(experiment_names=[ "DEMANDA-IA_FaseI_baseline_SKU_sucursal"])
padres = runs[runs["tags.mlflow.parentRunId"].isna()]
print(f"Corridas totales: {len(runs)} | padre/resumen: {len(padres)} | hijas de Grid Search: {len(runs)-len(padres)}")
padres[["tags.mlflow.runName", "tags.modelo", "params.mejor_max_depth", "params.mejor_alpha", "metrics.cv_mae_media", "metrics.test_mae", "metrics.test_rmse", "metrics.test_r2", "tags.git_commit", "tags.dataset_sha256"]].sort_values("tags.mlflow.runName")

In [ ]:
from mlflow.tracking import MlflowClient
cl = MlflowClient()
for v in cl.search_model_versions("name='demanda_ia_pronostico_sku_sucursal'"):
    print(f"versión {v.version}: algoritmo={v.tags.get('algoritmo')}  aliases={v.aliases}  run={v.run_id[:8]}")
campeon = mlflow.pyfunc.load_model("models:/demanda_ia_pronostico_sku_sucursal@campeon")
print("Prueba de carga del campeón desde el registro:", campeon.metadata.flavors.keys())

## 8. Conclusiones
Ver la sección de interpretación del reporte técnico (`reporte/`). En resumen: los tres modelos superan con claridad a la referencia naive; el ganador se elige por MAE en el holdout, respaldado por Friedman, Wilcoxon con corrección de Holm, Diebold–Mariano e intervalos bootstrap. El alias `campeon` del registro de MLflow permite que el proceso nocturno consuma siempre la versión aprobada sin cambiar código.